# VinDr-CXR: CLEAR with its own resize instead of padding (post hoc sensitivity check)

The CLEAR paper resizes images to 448 x 448 with bicubic interpolation, and the released `load_pretrained` preprocessing does the same (exact square, no padding). Our VinDr-CXR analysis (Amendment 2) padded non-square images to 448 instead. This notebook scores the same 15,000 images with CLEAR's own preprocessing and repeats the comparison with CheXzero, overall and in the subsets with and without recorded sex and age. CheXzero is not re-scored: its own preprocessing pads, and its saved scores are reused.

It first re-scores 256 images with the padded pipeline and checks that they match the saved scores, so that the comparison is like for like.

**Settings:** Accelerator **GPU T4 x2** (a single GPU is enough), Internet **On**.
**Inputs:** the competition **VinBigData Chest X-ray Abnormalities Detection**, and the output of your **vinbigdata_check** notebook (the version that produced the Amendment 2 results).
**Run:** Save Version, then Save & Run All (Commit). About 1.5 hours on GPU. At the end copy the block between `===== VINDR SQUASH SUMMARY =====` and `===== END =====`.


In [ ]:
# 0. Settings and packages
import os, sys, time, json, subprocess, platform
os.environ["HF_HOME"] = "/tmp/hf"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 2026; T0 = time.time()
OUT = "/kaggle/working/vin_squash_out"; os.makedirs(OUT, exist_ok=True)
!rm -rf /kaggle/working/CLEAR && git clone -q https://github.com/peterhan91/CLEAR.git /kaggle/working/CLEAR
!git -C /kaggle/working/CLEAR checkout -q 57d2eae57ddabf8e56c8013ec675336d71b3d8e7
!pip install -q /kaggle/working/CLEAR
!pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg
import numpy as np, pandas as pd, torch, torch.nn.functional as F, pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
run_info = {"started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "seed": SEED,
            "clear_git_commit": subprocess.run(["git", "-C", "/kaggle/working/CLEAR", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
            "python": platform.python_version(), "torch": torch.__version__, "pydicom": pydicom.__version__,
            "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
print(json.dumps(run_info, indent=1))


In [ ]:
# 1. Inputs (directory search skips image folders so it stays fast)
def walk_find(test):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in ("train", "test") and not d.startswith("images")]
        if test(root, files): return root
    return None
vin_dir = walk_find(lambda r, f: "train.csv" in f and os.path.isdir(os.path.join(r, "train")))
prev = walk_find(lambda r, f: "scores_chexzero.npy" in f and "clear_concept_projection_M.npy" in f)
assert vin_dir, "VinBigData competition data not found: add it under Input (Competitions)"
assert prev, "vinbigdata_check output not found: add that notebook under Input (Your Work)"
print("competition:", vin_dir, "| previous output:", prev)
VIN = {"Aortic enlargement": "aortic enlargement", "Atelectasis": "atelectasis", "Calcification": "calcification",
       "Cardiomegaly": "cardiomegaly", "Consolidation": "consolidation", "ILD": "interstitial lung disease",
       "Infiltration": "infiltration", "Lung Opacity": "lung opacity", "Nodule/Mass": "nodule or mass",
       "Pleural effusion": "pleural effusion", "Pleural thickening": "pleural thickening", "Pneumothorax": "pneumothorax",
       "Pulmonary fibrosis": "pulmonary fibrosis"}
FINDINGS = list(VIN); POS = [VIN[f] for f in FINDINGS]; NEG = ["no " + w for w in POS]
vc = pd.read_csv(os.path.join(prev, "cohort.csv"))
assert [c for c in vc.columns if c in VIN] == FINDINGS, "finding columns differ from the Amendment 2 run"
S_prev = {k: np.load(os.path.join(prev, f"scores_{k}.npy")).astype(np.float64) for k in ["chexzero", "clear_concept", "clear_direct"]}
assert all(v.shape == (len(vc), len(FINDINGS)) for v in S_prev.values())
M = torch.from_numpy(np.load(os.path.join(prev, "clear_concept_projection_M.npy"))).float()
pos_llm = np.load(os.path.join(prev, "prompt_emb_sfr_pos.npy")); neg_llm = np.load(os.path.join(prev, "prompt_emb_sfr_neg.npy"))
paths = [os.path.join(vin_dir, "train", i + ".dicom") for i in vc.image_id]
assert all(os.path.exists(p) for p in paths[:50])
run_info["n_images"] = len(vc); print("images:", len(vc), "| M", tuple(M.shape), "| prompts", pos_llm.shape)


In [ ]:
# 2. CLEAR, prompts and the two image pipelines
from clear import load_pretrained, tokenize
from PIL import Image
from torchvision.transforms import Normalize, ToTensor
from torch.utils.data import Dataset, DataLoader
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
clear_model, clear_pre = load_pretrained(device=dev); clear_model.eval()
run_info["clear_preprocess"] = repr(clear_pre)
M = M.to(dev)
pos_t = F.normalize(torch.from_numpy(pos_llm).float().to(dev), dim=-1); neg_t = F.normalize(torch.from_numpy(neg_llm).float().to(dev), dim=-1)
with torch.no_grad():
    cpos = F.normalize(clear_model.encode_text(tokenize(POS).to(dev)).float(), dim=-1)
    cneg = F.normalize(clear_model.encode_text(tokenize(NEG).to(dev)).float(), dim=-1)
def read_xray(p):                                   # identical to the Amendment 2 notebook
    d = pydicom.dcmread(p); a = d.pixel_array
    try: a = apply_voi_lut(a, d)
    except Exception: pass
    a = a.astype(np.float32)
    if getattr(d, "PhotometricInterpretation", "") == "MONOCHROME1": a = a.max() - a
    a -= a.min(); a /= max(float(a.max()), 1e-6)
    return Image.fromarray((a * 255).astype(np.uint8))
def pad_square(img, size):                          # identical to the Amendment 2 notebook
    w, h = img.size; r = float(size) / max(w, h)
    img = img.resize((max(1, int(w * r)), max(1, int(h * r))), Image.LANCZOS)
    out = Image.new("L", (size, size)); out.paste(img, ((size - img.size[0]) // 2, (size - img.size[1]) // 2))
    return out
cl_norm = Normalize((101.48761/255,)*3, (83.43944/255,)*3)
class DS(Dataset):
    def __init__(self, paths, mode): self.p, self.mode = list(paths), mode
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        try: im = read_xray(self.p[i]); ok = 1
        except Exception: im = Image.new("L", (448, 448)); ok = 0
        if self.mode == "pad": x = cl_norm(ToTensor()(pad_square(im, 448).convert("RGB")))
        else: x = clear_pre(im)                     # CLEAR's own preprocessing: RGB, exact 448 x 448 bicubic resize
        return x, ok
def pair_prob(f, tp, tn):
    p, n = f @ tp.T, f @ tn.T
    return torch.exp(p) / (torch.exp(p) + torch.exp(n))
def score(mode, idx, bs=32):
    dl = DataLoader(DS([paths[i] for i in idx], mode), batch_size=bs, num_workers=4, pin_memory=(dev != "cpu"))
    con, dirr, oks = [], [], []; t = time.time()
    with torch.no_grad():
        for b, (x, ok) in enumerate(dl):
            fi = F.normalize(clear_model.encode_image(x.to(dev)).float(), dim=-1)
            con.append(pair_prob(F.normalize(fi @ M, dim=-1), pos_t, neg_t).cpu().numpy())
            dirr.append(pair_prob(fi, cpos, cneg).cpu().numpy()); oks.append(ok.numpy())
            if b % 50 == 0: print(f"{mode}: {min((b+1)*bs, len(idx))}/{len(idx)} images, {round((time.time()-t)/60,1)} min", flush=True)
    return np.concatenate(con).astype(np.float64), np.concatenate(dirr).astype(np.float64), np.concatenate(oks).astype(bool)


In [ ]:
# 3. Check: the padded pipeline must reproduce the saved Amendment 2 scores (first 256 images)
chk = np.arange(256)
c_pad, d_pad, ok_pad = score("pad", chk)
run_info["repro_max_abs_diff"] = {"clear_concept": float(np.abs(c_pad - S_prev["clear_concept"][chk]).max()),
                                  "clear_direct": float(np.abs(d_pad - S_prev["clear_direct"][chk]).max())}
print("reproduction of saved padded scores, max abs difference:", run_info["repro_max_abs_diff"])
assert max(run_info["repro_max_abs_diff"].values()) < 1e-3, "padded pipeline does not reproduce the saved scores: stop and report"


In [ ]:
# 4. Score all images with CLEAR's own resize
c_sq, d_sq, ok_sq = score("squash", np.arange(len(vc)))
run_info["unreadable_images"] = int((~ok_sq).sum())
np.save(f"{OUT}/scores_clear_concept_squash.npy", c_sq.astype(np.float32)); np.save(f"{OUT}/scores_clear_direct_squash.npy", d_sq.astype(np.float32))
print("unreadable:", run_info["unreadable_images"])


In [ ]:
# 5. Analysis: same bootstrap (image level, 1,000 resamples, seed 2026) and subsets as the earlier VinDr analyses
class Cell:
    def __init__(self, s, y, pidx):
        o = np.argsort(s, kind="mergesort"); s = s[o]; self.y, self.p = y[o].astype(np.float64), pidx[o]
        self.starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    def auc(self, w=None):
        wi = 1.0 if w is None else w[self.p]
        bp = np.add.reduceat(wi * self.y, self.starts); bn = np.add.reduceat(wi * (1 - self.y), self.starts)
        P, N = bp.sum(), bn.sum()
        return np.nan if P == 0 or N == 0 else float((bp * (np.cumsum(bn) - bn + 0.5 * bn)).sum() / (P * N))
def ci(a, lvl=0.95):
    a = a[~np.isnan(a)]
    return (float("nan"), float("nan")) if a.size == 0 else (float(np.quantile(a, (1-lvl)/2)), float(np.quantile(a, 1-(1-lvl)/2)))
def table(s, Y, mask, pidx, WB):
    idx = np.flatnonzero(mask); pt, bt = [], []
    for j in range(Y.shape[1]):
        c = Cell(s[idx, j], Y[idx, j], pidx[idx]); pt.append(c.auc()); bt.append([c.auc(w) for w in WB])
    return np.array(pt), np.array(bt).T
def mac(tab, E): pt, bt = tab; return float(np.nanmean(pt[E])), np.nanmean(bt[:, E], axis=1)
def fmt(p, c): return f"{p:+.3f} ({c[0]:+.3f} to {c[1]:+.3f})"
Y = vc[FINDINGS].values; n = len(vc); iidx = np.arange(n)
rng = np.random.default_rng(SEED); WB = [np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float64) for _ in range(1000)]
SC = {"chexzero": S_prev["chexzero"], "concept_pad": S_prev["clear_concept"], "concept_squash": c_sq,
      "direct_pad": S_prev["clear_direct"], "direct_squash": d_sq}
sk = vc.sex.isin(["M", "F"]).values; ak = vc.age.notna().values
subsets = {"all": np.ones(n, bool), "sex_known": sk, "sex_missing": ~sk, "age_known": ak, "age_missing": ~ak}
RES = {}; LINES = []
for g, m in subsets.items():
    E = [j for j in range(len(FINDINGS)) if Y[m, j].sum() >= 30]
    T = {k: table(v, Y, m, iidx, WB) for k, v in SC.items()}
    pc, bc = mac(T["chexzero"], E); r = {"n": int(m.sum()), "eligible": [FINDINGS[j] for j in E], "chexzero": round(pc, 4)}
    line = f"-- {g} (n={int(m.sum())}, {len(E)} findings): CheXzero {pc:.3f}"
    for k in ["concept_pad", "concept_squash", "direct_pad", "direct_squash"]:
        pk, bk = mac(T[k], E)
        r[k] = {"macro": round(pk, 4), "minus_chexzero": [round(pk - pc, 4)] + [round(x, 4) for x in ci(bk - bc)],
                "wins_losses": [int((T[k][0][E] > T["chexzero"][0][E]).sum()), int((T[k][0][E] < T["chexzero"][0][E]).sum())]}
        line += f"\n     {k:<15} {pk:.3f} | minus CheXzero {fmt(pk - pc, ci(bk - bc))} | wins/losses {r[k]['wins_losses'][0]}/{r[k]['wins_losses'][1]}"
    for a, b in [("concept_squash", "concept_pad"), ("direct_squash", "direct_pad")]:
        pa, ba = mac(T[a], E); pb, bb = mac(T[b], E)
        r[f"{a}_minus_{b.split('_')[1]}"] = [round(pa - pb, 4)] + [round(x, 4) for x in ci(ba - bb)]
        line += f"\n     {a} minus {b}: {fmt(pa - pb, ci(ba - bb))}"
    r["per_finding"] = {FINDINGS[j]: {k: round(float(T[k][0][j]), 3) for k in SC} for j in E}
    RES[g] = r; LINES.append(line)
json.dump({"run_info": run_info, "results": RES}, open(f"{OUT}/vin_squash_results.json", "w"), indent=1, default=str)
!cd /kaggle/working && rm -f vin_squash_outputs.zip && zip -qr vin_squash_outputs.zip vin_squash_out
print("===== VINDR SQUASH SUMMARY =====")
print("images", n, "| unreadable", run_info["unreadable_images"], "| CLEAR commit", run_info["clear_git_commit"][:7])
print("CLEAR preprocessing:", run_info["clear_preprocess"].replace("\n", " "))
print("reproduction of saved padded scores (max abs diff):", run_info["repro_max_abs_diff"])
for l in LINES: print(l)
print("-- per finding, all images (CheXzero / concept pad / concept squash / direct pad / direct squash):")
for f, v in RES["all"]["per_finding"].items():
    print(f"   {f:<20} {v['chexzero']:.3f} / {v['concept_pad']:.3f} / {v['concept_squash']:.3f} / {v['direct_pad']:.3f} / {v['direct_squash']:.3f}")
print("minutes:", round((time.time() - T0) / 60, 1))
print("===== END =====")
